# **Importing packages that i will make use of.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# **Loading in my datasets.**
***The dataset contains more than one sheets, so i will save each one of them as a seperate table***

In [ ]:
product_table = pd.read_excel('/content/product_sales_diagnostic.xlsx', sheet_name='products')
product_table

,product_id,product_name,category,unit_price,launch_date,is_active
0,1,Wireless Earbuds Pro,Electronics,89.99,2022-01-15,1
1,2,Laptop Stand Aluminium,Accessories,45.00,2022-03-01,1
2,3,USB-C Hub 7-in-1,Accessories,35.50,2022-03-01,1
3,4,Mechanical Keyboard,Electronics,120.00,2022-05-10,1
4,5,Noise-Cancel Headphones,Electronics,199.99,2022-06-20,1
5,6,Webcam HD 1080p,Electronics,65.00,2022-07-01,1
6,7,Ergonomic Mouse,Accessories,55.00,2022-08-15,1
7,8,Monitor Arm Dual,Furniture,110.00,2022-09-01,1
8,9,Desk Organiser Set,Furniture,28.00,2022-10-01,1
9,10,Smart LED Desk Lamp,Furniture,42.00,2022-11-01,1


In [ ]:
order_table= pd.read_excel('/content/product_sales_diagnostic.xlsx', sheet_name='orders')
order_table.head()

,order_id,customer_id,order_date,region,channel,order_status
0,1001,201,2023-01-05,North,Online,Completed
1,1002,202,2023-01-08,South,Mobile App,Completed
2,1003,203,2023-01-12,East,Online,Completed
3,1004,204,2023-01-18,West,In-Store,Completed
4,1005,205,2023-01-22,North,Online,Completed


In [ ]:
item_table= pd.read_excel('/content/product_sales_diagnostic.xlsx', sheet_name='order_items')
item_table.head()

,item_id,order_id,product_id,quantity,unit_price,line_total
0,1,1001,1,2,89.99,179.98
1,2,1001,3,1,35.50,35.50
2,3,1002,5,1,199.99,199.99
3,4,1003,4,1,120.00,120.00
4,5,1003,2,1,45.00,45.00


# **Question: Why did revenue drop in Q4?**

***First, let's check if the theory is true by checking revenue over quarter of the year.***

In [ ]:
order_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   order_id      52 non-null     int64 
 1   customer_id   52 non-null     int64 
 2   order_date    52 non-null     object
 3   region        52 non-null     object
 4   channel       52 non-null     object
 5   order_status  52 non-null     object
dtypes: int64(2), object(4)
memory usage: 2.6+ KB


In [ ]:
# Covert oreder_date column to datetime format.
order_table['order_date'] = pd.to_datetime(order_table['order_date'])
order_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      52 non-null     int64         
 1   customer_id   52 non-null     int64         
 2   order_date    52 non-null     datetime64[ns]
 3   region        52 non-null     object        
 4   channel       52 non-null     object        
 5   order_status  52 non-null     object        
dtypes: datetime64[ns](1), int64(2), object(3)
memory usage: 2.6+ KB


In [ ]:
# Create quarter column
order_table['quarter'] = order_table['order_date'].dt.quarter
order_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   order_id      52 non-null     int64         
 1   customer_id   52 non-null     int64         
 2   order_date    52 non-null     datetime64[ns]
 3   region        52 non-null     object        
 4   channel       52 non-null     object        
 5   order_status  52 non-null     object        
 6   quarter       52 non-null     int32         
dtypes: datetime64[ns](1), int32(1), int64(2), object(3)
memory usage: 2.8+ KB


In [ ]:
# To access the revenue for each quarter, i will join order_table with item_table.
item_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   item_id     59 non-null     int64  
 1   order_id    59 non-null     int64  
 2   product_id  59 non-null     int64  
 3   quantity    59 non-null     int64  
 4   unit_price  59 non-null     float64
 5   line_total  59 non-null     float64
dtypes: float64(2), int64(4)
memory usage: 2.9 KB


In [ ]:
# Join the tables on matching column name
order_item= pd.merge(order_table,item_table, on= 'order_id', how= 'left')
order_item.head()

,order_id,customer_id,order_date,region,channel,order_status,quarter,item_id,product_id,quantity,unit_price,line_total
0,1001,201,2023-01-05,North,Online,Completed,1,1,1,2,89.99,179.98
1,1001,201,2023-01-05,North,Online,Completed,1,2,3,1,35.50,35.50
2,1002,202,2023-01-08,South,Mobile App,Completed,1,3,5,1,199.99,199.99
3,1003,203,2023-01-12,East,Online,Completed,1,4,4,1,120.00,120.00
4,1003,203,2023-01-12,East,Online,Completed,1,5,2,1,45.00,45.00


In [ ]:
# Now i want to see if really there was a drop in revenue in quarter 4

order_item.groupby('quarter')['line_total'].sum()

,line_total
quarter,
1,2200.42
2,1409.44
3,820.99
4,472.50


In [ ]:
# We can see that there is a whooping 79% decrease in revenue from quarter 1 to quarter 4. Now to the questions, why the drop in revenue?
order_item.head()

,order_id,customer_id,order_date,region,channel,order_status,quarter,item_id,product_id,quantity,unit_price,line_total
0,1001,201,2023-01-05,North,Online,Completed,1,1,1,2,89.99,179.98
1,1001,201,2023-01-05,North,Online,Completed,1,2,3,1,35.50,35.50
2,1002,202,2023-01-08,South,Mobile App,Completed,1,3,5,1,199.99,199.99
3,1003,203,2023-01-12,East,Online,Completed,1,4,4,1,120.00,120.00
4,1003,203,2023-01-12,East,Online,Completed,1,5,2,1,45.00,45.00


***Is it fewer orders or smaller orders? volume vs. basket size***

In [ ]:
# To check if it is fewer orders, i will count the number of unique order_id
order_item.groupby('quarter')['order_id'].nunique()

,order_id
quarter,
1,16
2,12
3,12
4,12


In [ ]:
# To examine if basket size
revenue= order_item.groupby('quarter')['line_total'].sum()
total_order= order_item.groupby('quarter')['order_id'].nunique()

basket_size= revenue/total_order
basket_size

,0
quarter,
1,137.526250
2,117.453333
3,68.415833
4,39.375000


***For my next analysis, i will check which category is driving the drop in revenue?Electronics vs. Accessories vs. Furniture***

In [ ]:
## Merge product_table with order_item and save in variable df

df= pd.merge(product_table,order_item, on= 'product_id', how= 'right')
df.head()

,product_id,product_name,category,unit_price_x,launch_date,is_active,order_id,customer_id,order_date,region,channel,order_status,quarter,item_id,quantity,unit_price_y,line_total
0,1,Wireless Earbuds Pro,Electronics,89.99,2022-01-15,1,1001,201,2023-01-05,North,Online,Completed,1,1,2,89.99,179.98
1,3,USB-C Hub 7-in-1,Accessories,35.50,2022-03-01,1,1001,201,2023-01-05,North,Online,Completed,1,2,1,35.50,35.50
2,5,Noise-Cancel Headphones,Electronics,199.99,2022-06-20,1,1002,202,2023-01-08,South,Mobile App,Completed,1,3,1,199.99,199.99
3,4,Mechanical Keyboard,Electronics,120.00,2022-05-10,1,1003,203,2023-01-12,East,Online,Completed,1,4,1,120.00,120.00
4,2,Laptop Stand Aluminium,Accessories,45.00,2022-03-01,1,1003,203,2023-01-12,East,Online,Completed,1,5,1,45.00,45.00


In [ ]:
# Filter df table for completed orders only
completed_order= df[df['order_status']=='Completed']

In [ ]:
# Group by both quarter and category and calculate total revenue (line_total) per group
completed_order.groupby(['quarter','category'])['line_total'].sum()

quarter  category   
1        Accessories     350.50
         Electronics    1329.93
         Furniture       236.00
2        Accessories     188.50
         Electronics     959.94
         Furniture       110.00
3        Accessories     275.00
         Electronics      65.00
         Furniture       166.00
4        Accessories     118.50
Name: line_total, dtype: float64

## ***Why did Electronic stop selling?***

***My first step to ananlysing this is to check electronics orders got cancelled or returned?***

In [ ]:
# Count the unique electronic order that was cancelled and returned
df[(df['category']=='Electronics')].groupby(['quarter','order_status'])['order_id'].nunique()

quarter  order_status
1        Completed       10
         Returned         1
2        Completed        6
         Returned         1
3        Completed        1
         Returned         2
Name: order_id, dtype: int64

***My next stop is to check if electronic was covered by discount in Q4***

In [ ]:
# First thing first, i will load in my discount table with table df

discount_table= pd.read_excel('/content/product_sales_diagnostic.xlsx', sheet_name= 'discounts')
discount_table.head(5)

,discount_id,product_id,campaign_name,discount_pct,start_date,end_date
0,1,1,New Year Electronics Sale,10,2023-01-01,2023-01-15
1,2,5,New Year Electronics Sale,10,2023-01-01,2023-01-15
2,3,4,Valentine Tech Bundle,12,2023-02-10,2023-02-14
3,4,6,Spring Work-from-Home Promo,15,2023-03-01,2023-03-31
4,5,2,Spring Work-from-Home Promo,15,2023-03-01,2023-03-31


In [ ]:
# Merge my df table with discount_table
df2= pd.merge(df,discount_table, on= "product_id", how='left')
df2.head()

,product_id,product_name,category,unit_price_x,launch_date,is_active,order_id,customer_id,order_date,region,...,quarter,item_id,quantity,unit_price_y,line_total,discount_id,campaign_name,discount_pct,start_date,end_date
0,1,Wireless Earbuds Pro,Electronics,89.99,2022-01-15,1,1001,201,2023-01-05,North,...,1,1,2,89.99,179.98,1.0,New Year Electronics Sale,10.0,2023-01-01,2023-01-15
1,1,Wireless Earbuds Pro,Electronics,89.99,2022-01-15,1,1001,201,2023-01-05,North,...,1,1,2,89.99,179.98,11.0,Back-to-School Tech,10.0,2023-08-15,2023-08-31
2,3,USB-C Hub 7-in-1,Accessories,35.50,2022-03-01,1,1001,201,2023-01-05,North,...,1,2,1,35.50,35.50,6.0,Spring Work-from-Home Promo,15.0,2023-03-01,2023-03-31
3,5,Noise-Cancel Headphones,Electronics,199.99,2022-06-20,1,1002,202,2023-01-08,South,...,1,3,1,199.99,199.99,2.0,New Year Electronics Sale,10.0,2023-01-01,2023-01-15
4,5,Noise-Cancel Headphones,Electronics,199.99,2022-06-20,1,1002,202,2023-01-08,South,...,1,3,1,199.99,199.99,12.0,Q4 Clearance - No Traction,5.0,2023-10-01,2023-10-31


In [ ]:
# Check average discount on electronic per quarters

df2[df2['category']=='Electronics'].groupby('quarter')['discount_pct'].mean()

,discount_pct
quarter,
1,9.950000
2,10.583333
3,13.750000


***After uncovering that electronics was not rescued by discounts despite increasing discount rates through Q3, completed orders kept falling and returns kept rising. By Q4, no discount was even offered, and electronics disappeared from orders entirely. I will go ahead to carryout product-level breakdown. This should tell me whether this was a whole-category problem or one or two specific products that stopped selling.***

In [ ]:
df[df['category'] == 'Electronics'].groupby(['quarter', 'product_name'])['line_total'].sum()

quarter  product_name           
1        Gaming Headset RGB         149.99
         Mechanical Keyboard        240.00
         Noise-Cancel Headphones    399.98
         Portable SSD 1TB            95.00
         Webcam HD 1080p            195.00
         Wireless Earbuds Pro       449.95
2        Gaming Headset RGB         149.99
         Mechanical Keyboard        120.00
         Noise-Cancel Headphones    199.99
         Portable SSD 1TB            95.00
         Webcam HD 1080p            130.00
         Wireless Earbuds Pro       359.96
3        Gaming Headset RGB         149.99
         Portable SSD 1TB            95.00
         Webcam HD 1080p             65.00
Name: line_total, dtype: float64

## **The Electronics collapse was not caused by one weak product, it was a category-wide failure. The three highest-revenue Electronics products (Earbuds, Headphones, Keyboard) disappeared after Q2, and the remaining products faded through Q3 before the entire category went silent in Q4. This single category accounted for the majority of the basket size decline.**